# Step 2: Sentiment Analysis
## YouTube Fast Fashion Intelligence Engine — CSCI370

**What is Sentiment Analysis?**

Sentiment analysis is the process of automatically detecting the *emotional tone* of a piece of text.
For each YouTube comment, we want to know: is this person expressing a **Positive**, **Negative**, or **Neutral** opinion?

**Why VADER?**

We use a tool called **VADER** (Valence Aware Dictionary and sEntiment Reasoner).
It was specifically designed for social media text — it understands things like:
- ALL CAPS ("this is AWFUL") → more negative
- Punctuation ("great!!!") → more positive  
- Emojis and slang

VADER gives every comment a **compound score** between -1.0 (most negative) and +1.0 (most positive).
We then convert that score into a simple label: Positive, Negative, or Neutral.


In [ ]:
# Install the VADER sentiment library
# Only needs to be run once
!pip install vaderSentiment -q

## 1. Load Libraries and Data

In [3]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Load the cleaned dataset from Step 1
df = pd.read_csv('youtube_comments_40k_cleaned.csv')

print(f"Loaded {len(df):,} comments")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

Loaded 39,422 comments
Columns: ['author', 'updated_at', 'like_count', 'text', 'video_id', 'public', 'text_clean']


,author,updated_at,like_count,text,video_id,public,text_clean
0,@joaquinrodriguezvillegas366,2026-02-25 00:49:20+00:00,0,This doesn't surprise me since the entire hist...,qpClEvsjW0s,True,This doesn't surprise me since the entire hist...
1,@andrerosekriel1127,2026-02-22 11:15:55+00:00,0,Okay then why work for these factories.....or ...,qpClEvsjW0s,True,Okay then why work for these factories.....or ...
2,@sbradshaw1886,2026-03-17 02:12:15+00:00,0,Please do not be this ignorant. The US and oth...,qpClEvsjW0s,True,Please do not be this ignorant. The US and oth...


## 2. Understanding VADER Scores

Before running on the full dataset, let's see exactly what VADER returns so we understand the output.

VADER gives 4 scores per comment:
- `neg` → proportion of negative sentiment (0 to 1)
- `neu` → proportion of neutral sentiment (0 to 1)  
- `pos` → proportion of positive sentiment (0 to 1)
- `compound` → the **overall score** from -1.0 to +1.0 (this is the one we use)

**Our labeling rule (industry standard thresholds):**
- compound >= 0.05 → **Positive**
- compound <= -0.05 → **Negative**
- anything in between → **Neutral**


In [4]:
# Create the VADER analyzer object
analyzer = SentimentIntensityAnalyzer()

# Test it on 3 example comments so we can see exactly how it works
examples = [
    "Shein is absolutely destroying the environment and exploiting workers",
    "I love buying from Shein, the prices are amazing!",
    "It is what it is, just another fast fashion brand"
]

print("VADER score breakdown:\n")
for text in examples:
    scores = analyzer.polarity_scores(text)
    print(f"Text   : {text}")
    print(f"Scores : {scores}")
    print(f"Label  : {'Positive' if scores['compound'] >= 0.05 else 'Negative' if scores['compound'] <= -0.05 else 'Neutral'}")
    print("-" * 70)

VADER score breakdown:

Text   : Shein is absolutely destroying the environment and exploiting workers
Scores : {'neg': 0.492, 'neu': 0.508, 'pos': 0.0, 'compound': -0.7778}
Label  : Negative
----------------------------------------------------------------------
Text   : I love buying from Shein, the prices are amazing!
Scores : {'neg': 0.0, 'neu': 0.458, 'pos': 0.542, 'compound': 0.8516}
Label  : Positive
----------------------------------------------------------------------
Text   : It is what it is, just another fast fashion brand
Scores : {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
Label  : Neutral
----------------------------------------------------------------------


## 3. Run Sentiment Analysis on All 39,422 Comments

We apply VADER to every row in the dataset. This takes about 5 seconds.

We will add 2 new columns to our dataframe:
- `sentiment_score` → the raw compound number (-1.0 to +1.0)
- `sentiment_label` → the human-readable label (Positive / Negative / Neutral)


In [5]:
# Step 1: Get the compound score for every comment
# We use .apply() which runs our function on each row one by one
df['sentiment_score'] = df['text_clean'].apply(
    lambda text: analyzer.polarity_scores(str(text))['compound']
)

# Step 2: Convert the score into a label using our thresholds
def score_to_label(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment_label'] = df['sentiment_score'].apply(score_to_label)

print("Sentiment analysis complete!")
print(f"\nSentiment distribution:")
print(df['sentiment_label'].value_counts())
print(f"\nAs percentages:")
print(df['sentiment_label'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

Sentiment analysis complete!

Sentiment distribution:
sentiment_label
Positive    16707
Negative    11889
Neutral     10826
Name: count, dtype: int64

As percentages:
sentiment_label
Positive    42.4%
Negative    30.2%
Neutral     27.5%
Name: proportion, dtype: object


## 4. Sanity Check — Do the Labels Make Sense?

Let's look at real examples of comments that were labelled Positive, Negative, and Neutral
to make sure VADER is working correctly on our data.


In [6]:
# Show 3 real examples from each sentiment category
for label in ['Positive', 'Negative', 'Neutral']:
    print(f"\n{'='*60}")
    print(f"  {label.upper()} EXAMPLES")
    print(f"{'='*60}")
    sample = df[df['sentiment_label'] == label][['text_clean', 'sentiment_score']].sample(3, random_state=42)
    for _, row in sample.iterrows():
        print(f"  Score: {row['sentiment_score']:.3f}")
        print(f"  Text : {row['text_clean'][:200]}")
        print()


  POSITIVE EXAMPLES
  Score: 0.612
  Text : Lol yeah, though tbf Hollister is a MUCH larger company and so, in many ways, should be able to afford sell their clothes for less.

  Score: 0.511
  Text : @Marzxh3l they are. Small businesses and individual boutiques that aren't part of chain stores are ethical. If I'm shopping online I usually go to Etsy or someone's Shopify store because I know they m

  Score: 0.273
  Text : Well atleast they dont throw it away so


  NEGATIVE EXAMPLES
  Score: -0.735
  Text : the gravity of this youtube comment is shocking to me. how horrible of a realization

  Score: -0.146
  Text : You guys should know about Bangladesh and Cambodia, their monthly salary is only 300$ a month, the working environment can't even guarantee safety like here.

  Score: -0.652
  Text : TODOS, TODOS, TODAS, TODAS RESPONSABLES, SOMOS RESPONSABLES. SIN EXCEPCIÓN. ☠☠☠☠☠☠


  NEUTRAL EXAMPLES
  Score: 0.000
  Text : Que triste es ver estos videos da tanta indignacion como les r

## 5. Sentiment Breakdown Per Video

Since our dataset has 24 different videos, let's see if the sentiment differs
across videos. This will be a key insight for the dashboard.


In [7]:
# Group by video_id and count each sentiment label
sentiment_by_video = df.groupby(['video_id', 'sentiment_label']).size().unstack(fill_value=0)

# Add a total column and sort by total comments
sentiment_by_video['total'] = sentiment_by_video.sum(axis=1)
sentiment_by_video = sentiment_by_video.sort_values('total', ascending=False)

print("Sentiment counts per video (top 10 by comment volume):")
print(sentiment_by_video.head(10).to_string())

Sentiment counts per video (top 10 by comment volume):
sentiment_label  Negative  Neutral  Positive  total
video_id                                           
LWvOlZ4hPU0          3941     4604      3174  11719
_KKhXQ9Iz-U          1912     1928      3553   7393
fd228YQPn-0          2169     1433      3244   6846
U4km0Cslcpg          1376      873      2288   4537
XT6Hwx20m5M           678      295      1105   2078
VaS-iVwaOLw           280      288       539   1107
23vUvQN-R1Y           346      191       449    986
iq0--DfC2Xk           160      161       384    705
tLfNUD0-8ts           154      210       341    705
04dSAUAoitY           175      144       341    660


## 6. Intermediate Check — VADER Results So Far

Before moving to the ensemble step, let's verify VADER is working correctly
on the clear-cut cases. **We do NOT save here** — the final save happens after
RoBERTa has processed the borderline comments in Section 7.


In [8]:
# ── Intermediate check only — DO NOT save here ───────────────────────────────
# The CSV is saved at the end of Section 7 after RoBERTa has run.
# Saving here would give you an incomplete VADER-only file.

# Quick check: run VADER scores and preview results
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

df['sentiment_score'] = df['text_clean'].apply(
    lambda t: analyzer.polarity_scores(str(t))['compound']
)

def score_to_label(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment_label'] = df['sentiment_score'].apply(score_to_label)

print("VADER intermediate results (NOT final — RoBERTa still needs to run on borderline cases)")
print(f"\nSentiment distribution so far:")
print(df['sentiment_label'].value_counts())
print(f"\nAs percentages:")
print(df['sentiment_label'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')
print("\n⚠️  Continue to Section 7 to run RoBERTa and get the final labels.")

VADER intermediate results (NOT final — RoBERTa still needs to run on borderline cases)

Sentiment distribution so far:
sentiment_label
Positive    16707
Negative    11889
Neutral     10826
Name: count, dtype: int64

As percentages:
sentiment_label
Positive    42.4%
Negative    30.2%
Neutral     27.5%
Name: proportion, dtype: object

⚠️  Continue to Section 7 to run RoBERTa and get the final labels.


## Checkpoint — VADER Done, RoBERTa Next

**What VADER did:**
- Scored all 30,609 comments with a compound value from -1.0 to +1.0
- Assigned a preliminary label: Positive / Negative / Neutral

**What happens next in Section 7:**
- Comments with |compound| ≥ 0.2 → keep their VADER label (confident cases)
- Comments with |compound| < 0.2 → re-scored by RoBERTa (borderline cases)
- Final CSV saved only after RoBERTa finishes

⚠️ **Do not stop here — the file has not been saved yet. Continue to Section 7.**


---
## 7. Ensemble Sentiment — RoBERTa for Borderline Cases

### Why do we need a second model?

VADER is a **lexicon-based** model — it looks up each word in a dictionary of sentiment scores
and adds them up. This works well for clear cases but fails when:
- **Sarcasm is used** — *"Please do not be this ignorant"* → VADER sees "please" (positive) and gets confused
- **Context matters** — *"first-world problems"* has no sentiment score in VADER's dictionary
- **Negation is complex** — longer sentences where "not" applies across many words

So we use an **ensemble approach**:
- ✅ VADER handles the **clear cases** (compound ≥ 0.2 or ≤ -0.2) — fast, reliable
- 🤖 RoBERTa handles the **borderline cases** (compound between -0.2 and +0.2) — slower but understands context

**What is RoBERTa?**
`cardiffnlp/twitter-roberta-base-sentiment` is a transformer model trained on **58 million tweets**.
Unlike VADER which looks up words, RoBERTa reads the *entire sentence at once* and understands
relationships between words — so it knows "please do not be ignorant" is negative even though
"please" alone is positive.

**The threshold we use: ±0.2 (not ±0.05)**
We widen the borderline zone to ±0.2 because we want to catch all cases where
VADER might be wrong, not just the ones right at the boundary.
In our dataset that is **~8,276 comments (27%)** that go to RoBERTa.


In [9]:
# Install the transformers library (needed for RoBERTa)
!pip install transformers torch -q


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from transformers import pipeline

# Load the RoBERTa model
# This downloads ~500MB on first run — subsequent runs use the cached version
print("Loading RoBERTa model... (first run downloads ~500MB, be patient)")

roberta = pipeline(
    task='sentiment-analysis',
    model='cardiffnlp/twitter-roberta-base-sentiment-latest',
    tokenizer='cardiffnlp/twitter-roberta-base-sentiment-latest',
    max_length=512,
    truncation=True      # YouTube comments can be long — truncate if over 512 tokens
)

print("RoBERTa loaded!")

# Test it on the exact comment that VADER got wrong
test = "Please do not be this ignorant. The US and other countries are called first-world"
result = roberta(test)
print(f"\nTest comment : {test}")
print(f"RoBERTa says : {result}")

C:\Users\AK Khan\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading RoBERTa model... (first run downloads ~500MB, be patient)


C:\Users\AK Khan\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\AK Khan\.cache\huggingface\hub\models--cardiffnlp--twitter-roberta-base-sentiment-latest. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10889.93it/s]
[t

RoBERTa loaded!

Test comment : Please do not be this ignorant. The US and other countries are called first-world
RoBERTa says : [{'label': 'negative', 'score': 0.808952271938324}]


### Run the Ensemble Pipeline

**Step 1:** VADER scores all 30,609 comments
**Step 2:** Clear cases (|compound| ≥ 0.2) keep their VADER label
**Step 3:** Borderline cases (|compound| < 0.2) are re-scored by RoBERTa
**Step 4:** Final label = VADER label OR RoBERTa label depending on confidence


In [11]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

analyzer = SentimentIntensityAnalyzer()

# ── Step 1: VADER scores everything ───────────────────────────────────────────
print("Step 1: Running VADER on all comments...")
df['sentiment_score'] = df['text_clean'].apply(
    lambda t: analyzer.polarity_scores(str(t))['compound']
)

# ── Step 2: Split into clear vs borderline ────────────────────────────────────
# Threshold of ±0.2 means VADER needs to be fairly confident to keep its label
THRESHOLD = 0.2

clear      = df[abs(df['sentiment_score']) >= THRESHOLD].copy()
borderline = df[abs(df['sentiment_score']) <  THRESHOLD].copy()

print(f"Step 2: Split complete")
print(f"  Clear cases (VADER confident) : {len(clear):,}  ({len(clear)/len(df)*100:.1f}%)")
print(f"  Borderline (goes to RoBERTa)  : {len(borderline):,}  ({len(borderline)/len(df)*100:.1f}%)")

Step 1: Running VADER on all comments...
Step 2: Split complete
  Clear cases (VADER confident) : 26,110  (66.2%)
  Borderline (goes to RoBERTa)  : 13,312  (33.8%)


In [12]:
# ── Step 3: Label clear cases with VADER ─────────────────────────────────────
def vader_label(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

clear['sentiment_label'] = clear['sentiment_score'].apply(vader_label)
clear['sentiment_source'] = 'VADER'   # track which model gave the label

# ── Step 4: Re-score borderline cases with RoBERTa ───────────────────────────
# RoBERTa returns labels 'positive', 'negative', 'neutral' — we capitalise them

print("Step 3: Running RoBERTa on borderline comments...")
print("(This may take 10-20 minutes depending on your machine)")

# RoBERTa can only process 512 tokens at a time — we truncate long comments
def roberta_label(text):
    try:
        result = roberta(str(text)[:512])[0]
        return result['label'].capitalize()
    except Exception:
        # If RoBERTa fails on a comment, fall back to VADER label
        score = analyzer.polarity_scores(str(text))['compound']
        return vader_label(score)

borderline['sentiment_label']  = borderline['text_clean'].apply(roberta_label)
borderline['sentiment_source'] = 'RoBERTa'

print("RoBERTa scoring complete!")

Step 3: Running RoBERTa on borderline comments...
(This may take 10-20 minutes depending on your machine)
RoBERTa scoring complete!


In [13]:
# ── Step 5: Merge back into one dataframe ────────────────────────────────────
df_final = pd.concat([clear, borderline]).sort_index()

print("Final ensemble sentiment distribution:")
print(df_final['sentiment_label'].value_counts())
print()
print("Labels by source model:")
print(df_final.groupby('sentiment_source')['sentiment_label'].value_counts().to_string())

Final ensemble sentiment distribution:
sentiment_label
Positive    16737
Negative    14261
Neutral      8424
Name: count, dtype: int64

Labels by source model:
sentiment_source  sentiment_label
RoBERTa           Neutral             8424
                  Negative            3622
                  Positive            1266
VADER             Positive           15471
                  Negative           10639


In [14]:
# ── Step 6: Spot-check — did RoBERTa fix the problem cases? ─────────────────
print("Spot-check — comments where RoBERTa overruled VADER:")
print("=" * 70)

# Show examples where VADER was near-neutral but RoBERTa gave a strong label
checks = df_final[
    (df_final['sentiment_source'] == 'RoBERTa') &
    (df_final['sentiment_label'] != 'Neutral')
][['text_clean', 'sentiment_score', 'sentiment_label']].sample(8, random_state=42)

for _, row in checks.iterrows():
    vader_lbl = vader_label(row['sentiment_score'])
    print(f"\n  Text        : {str(row['text_clean'])[:120]}")
    print(f"  VADER score : {row['sentiment_score']:.3f}  → would have been '{vader_lbl}'")
    print(f"  RoBERTa     : '{row['sentiment_label']}'")

Spot-check — comments where RoBERTa overruled VADER:

  Text        : Marjan nobody is gonna spend 20 dollars on a fucking tee shirt
  VADER score : 0.000  → would have been 'Neutral'
  RoBERTa     : 'Negative'

  Text        : This gave me aids
  VADER score : 0.000  → would have been 'Neutral'
  RoBERTa     : 'Negative'

  Text        : Only Real one recognizes the Real Op that Swynney is, you simps gotta grow up.
  VADER score : 0.000  → would have been 'Neutral'
  RoBERTa     : 'Negative'

  Text        : Complimenti Giuseppe da papà mi sono messo a piangere....mi sono medesimato in queste famiglie con figli destinati ad un
  VADER score : 0.000  → would have been 'Neutral'
  RoBERTa     : 'Positive'

  Text        : Sorry about your divorce…
  VADER score : -0.077  → would have been 'Negative'
  RoBERTa     : 'Negative'

  Text        : I bought from Shein once. I'm a 54 yo gay man. It was cheap, with cheap quality. Never bought anything from it again.
  VADER score : 0.000  → wou

In [15]:
# ── Step 7: Save the final ensemble result ───────────────────────────────────
output_path = 'youtube_comments_sentiment.csv'
df_final.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Total rows     : {len(df_final):,}")
print(f"New columns    : sentiment_score, sentiment_label, sentiment_source")
print(f"\nPreview:")
df_final[['text_clean', 'sentiment_score', 'sentiment_label', 'sentiment_source']].head(5)

Saved to: youtube_comments_sentiment.csv
Total rows     : 39,422
New columns    : sentiment_score, sentiment_label, sentiment_source

Preview:


,text_clean,sentiment_score,sentiment_label,sentiment_source
0,This doesn't surprise me since the entire hist...,-0.0121,Negative,RoBERTa
1,Okay then why work for these factories.....or ...,-0.2732,Negative,VADER
2,Please do not be this ignorant. The US and oth...,-0.0194,Negative,RoBERTa
3,"@sbradshaw1886 Still a choice ...Ignorence, re...",0.0000,Neutral,RoBERTa
4,They fuel buying addiction. Same with Temu. No...,-0.1531,Negative,RoBERTa


### Summary of Ensemble Approach

| Model | Role | Speed | Strength |
|---|---|---|---|
| **VADER** | Scores all comments first, keeps clear cases | Very fast (~5 sec) | Great for obvious positive/negative social media text |
| **RoBERTa** | Re-scores borderline cases only | Slower (~15 min) | Understands context, sarcasm, complex sentences |

**New column added: `sentiment_source`**
This tells you which model gave the final label for each comment — useful for your report
to show exactly how many comments each model handled.

**Why this matters for your report:**
This is called a **hybrid / ensemble NLP pipeline** — using multiple models where each
covers the other's weakness. This is a genuinely sophisticated design decision that goes
beyond a basic student project.
